# AFC_Controller

### --->--->---> THIS WAS MADE USING AI AND NEEDS TO BE REDONE. THIS IS JUST FOR AN EXAMPLE <---<---<---

The air-brake (AFC) control law, kept separate from the rocket definition so the control logic can be changed or swapped without touching the airframe. It defines `controller_function`, which `Method_of_Sections` passes to `rocket.add_air_brakes()`.

**Why this works as a separate file:** the function only *refers to* `rocket`, `M2500T` and the parameters. Python looks those names up when the function is **called** during the flight, not when it's defined, so this notebook can be loaded before the rocket exists.

**To try a different controller:** copy this notebook (e.g. `AFC_Controller_PID`), keep the same function name and signature, and change the `%run` line in `Test_Flight_MoS` to load the new one.

### How this notebook loads its inputs
This notebook doesn't redefine any numbers; it reads them from the notebooks it depends on.
- **Opened on its own:** the cell below runs its dependencies with IPython's `%run` (written as `get_ipython().run_line_magic("run", ...)` because the call sits inside an `if`). When `%run` runs a notebook it executes in the **same namespace**, so every variable it defines becomes available here.
- **Run from `Test_Flight_MoS`:** that notebook sets `PIPELINE = True` and has already loaded everything, so this cell skips the reload. `SHOW_INFO` turns off the `.info()` printouts and plots so the test flight output stays clean.

In [1]:
from rocketpy import Rocket, SolidMotor, Environment, Flight   # simulation classes
import numpy as np                                              # arrays and maths
import pandas as pd                                             # reading .xlsx data files
from datetime import datetime, timedelta                                                 # launch date for the forecast
import matplotlib.pyplot as plt                                 # plotting

In [2]:
# PIPELINE is set to True by Test_Flight_MoS before it runs this notebook.
#   True  -> dependencies are already loaded; skip the info printouts and plots
#   False -> this notebook was opened on its own, so load what it needs first
PIPELINE = globals().get("PIPELINE", False)
if not PIPELINE:
    get_ipython().run_line_magic("run", "25_26_MoS_Parameters.ipynb")   # load 25_26_MoS_Parameters
SHOW_INFO = not PIPELINE   # only print .info() / draw plots when run on its own

## Control law
RocketPy calls `controller_function` at `AIRBRAKE_SAMPLING_RATE` (10 Hz) during the flight, the way your flight computer would run its control loop. On each call it does five things:
1. **Reads the state:** the position and velocity vector, `state = [x, y, z, vx, vy, vz, ...]`.
2. **Finds the Mach number** from the airspeed (rocket velocity minus wind) and the local speed of sound.
3. **Stays closed during the burn and after apogee.** The brakes only work between burnout and apogee.
4. **Predicts apogee** if the brakes stay where they are, using the exact solution for a vertical coast with quadratic drag:

   $$h_{apo} = h + \frac{m}{2k}\ln\!\left(1+\frac{k\,v_z^2}{m\,g}\right), \qquad k = \tfrac{1}{2}\rho\,(C_{D,rocket}+C_{D,brakes})\,A_{ref}$$

   It treats the coast as vertical and freezes ρ and Cd at their current values. That's conservative early in the coast, because the air thins as the rocket climbs, but the prediction improves as apogee gets closer.
5. **Sets the new deployment** with a proportional command (0.002 per metre of predicted overshoot). The change is limited to `MAX_RATE / sampling_rate` per step, to match how fast the actuator can move, and the level is kept between 0 and 1.

**Return value:** `(time, deployment, Cd)`. RocketPy saves it every step, and `Test_Flight_MoS` reads it back with `get_controller_observed_variables()` for the deployment and Cd plots.

**Limitation:** RocketPy's air brakes add drag only. They don't move the CP or reduce the fins' effectiveness, so the brakes' effect on stability seen in CFD isn't captured.

In [3]:
def controller_function(time, sampling_rate, state, state_history, observed_variables,
                        air_brakes, sensors, environment):
    # state = [x, y, z, vx, vy, vz, ...]; state_history rows = [t, x, y, z, vx, vy, vz, ...]
    altitude_ASL = state[2]                        # m above sea level
    vx, vy, vz = state[3], state[4], state[5]      # m/s, inertial frame

    # Airspeed = rocket velocity relative to the wind -> Mach number
    wind_x, wind_y = environment.wind_velocity_x(altitude_ASL), environment.wind_velocity_y(altitude_ASL)
    free_stream_speed = ((wind_x - vx)**2 + (wind_y - vy)**2 + vz**2) ** 0.5
    mach_number = free_stream_speed / environment.speed_of_sound(altitude_ASL)

    if time < M2500T.burn_out_time or vz <= 0:     # only between burnout and apogee
        air_brakes.deployment_level = 0
        return time, 0, 0

    # Predicted apogee with the brakes held where they are (quadratic drag, vertical coast)
    rho  = environment.density(altitude_ASL)                        # kg/m^3
    mass = rocket.total_mass(time)                                  # kg (motor is burnt out)
    cd   = np.interp(mach_number, CD_ROCKET[:, 0], CD_ROCKET[:, 1]) \
           + air_brakes.drag_coefficient(air_brakes.deployment_level, mach_number)   # rocket + brakes
    k = 0.5 * rho * cd * np.pi * BODY_RADIUS**2                     # drag constant, F = k v^2
    apogee_pred = altitude_ASL + mass / (2 * k) * np.log(1 + k * vz**2 / (mass * G)) - environment.elevation
    error = apogee_pred - TARGET_APOGEE_FT / 3.28084                # [m], > 0 means too high

    # Proportional command, rate-limited like the docs example
    new_level = air_brakes.deployment_level + 0.002 * error         # gain: 0.002 per metre of overshoot
    max_change = MAX_RATE / sampling_rate                           # actuator speed limit per step
    new_level = min(max(new_level, air_brakes.deployment_level - max_change),
                    air_brakes.deployment_level + max_change)
    air_brakes.deployment_level = min(max(new_level, 0), 1)         # keep between closed and fully open

    # Saved every step -> test_flight.get_controller_observed_variables()
    return (time, air_brakes.deployment_level,
            air_brakes.drag_coefficient(air_brakes.deployment_level, mach_number))

## Function signature
RocketPy always calls the controller with these eight arguments, in this order, so the signature must stay the same even if some arguments go unused:

| Argument | What it is |
|---|---|
| `time` | simulation time (s) |
| `sampling_rate` | controller rate (Hz), from `AIRBRAKE_SAMPLING_RATE` |
| `state` | current state `[x, y, z, vx, vy, vz, e0, e1, e2, e3, wx, wy, wz]` |
| `state_history` | all previous states, for filtering or derivative terms |
| `observed_variables` | everything this function returned on earlier calls |
| `air_brakes` | the AirBrakes object; set `air_brakes.deployment_level` here |
| `sensors` | simulated sensors attached to the rocket (none yet) |
| `environment` | `env`, for density, wind and speed of sound |

Whatever the function returns is saved each step and read back in `Test_Flight_MoS` with `get_controller_observed_variables()`.